In [119]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
import pickle
import warnings
warnings.filterwarnings('ignore')
from imblearn.over_sampling import SMOTE

 # STEP 1 : Data Cleaning and Preprocessing

In [120]:
df = pd.read_csv('../data/DNS_dataset.csv')
print(f'\nInitial shape: {df.shape}')
print(f'Label distribution:\n{df["Label"].value_counts()}')


Initial shape: (287196, 37)
Label distribution:
Label
Normal    207355
Tunnel     79841
Name: count, dtype: int64


In [121]:
df.columns

Index(['SourceIP', 'DestIP', 'SourcePort', 'DestPort', 'Label', 'WindowID',
       'Character_frequency_entropy', 'Domain_name_length',
       'Non_printable_character_count', 'Digit_letter_ratio',
       'Unique_subdomains_per_flow', 'NXDomain_ratio', 'Query_rate_per_sec',
       'Response_size_variance', 'Duration', 'FlowBytesSent', 'FlowSentRate',
       'FlowBytesReceived', 'FlowReceivedRate', 'PacketLengthVariance',
       'PacketLengthStandardDeviation', 'PacketLengthMean',
       'PacketLengthMedian', 'PacketLengthMode', 'PacketLengthSkewFromMedian',
       'PacketLengthSkewFromMode', 'PacketLengthCoefficientofVariation',
       'PacketTimeStandardDeviation', 'PacketTimeMedian',
       'PacketTimeSkewFromMedian', 'PacketTimeSkewFromMode',
       'PacketTimeCoefficientofVariation', 'ResponseTimeTimeStandardDeviation',
       'ResponseTimeTimeMean', 'ResponseTimeTimeMedian',
       'ResponseTimeTimeSkewFromMedian', 'ResponseTimeTimeSkewFromMode'],
      dtype='object')

In [122]:
df.head()

,SourceIP,DestIP,SourcePort,DestPort,Label,WindowID,Character_frequency_entropy,Domain_name_length,Non_printable_character_count,Digit_letter_ratio,...,PacketTimeStandardDeviation,PacketTimeMedian,PacketTimeSkewFromMedian,PacketTimeSkewFromMode,PacketTimeCoefficientofVariation,ResponseTimeTimeStandardDeviation,ResponseTimeTimeMean,ResponseTimeTimeMedian,ResponseTimeTimeSkewFromMedian,ResponseTimeTimeSkewFromMode
0,192.168.117.128,192.168.117.2,33729.0,53.0,Tunnel,0,3.722907,28,0,1.2000,...,0.032117,0.000000,0.022710,0.022710,1.414214,0.032117,0.022710,0.000000,0.022710,0.022710
1,192.168.117.128,192.168.117.2,35730.0,53.0,Tunnel,0,3.532573,26,0,1.0000,...,0.003677,0.000000,0.002600,0.002600,1.414214,0.003677,0.002600,0.000000,0.002600,0.002600
2,192.168.117.128,192.168.117.2,38462.0,53.0,Tunnel,0,3.594466,21,0,0.0625,...,0.015910,0.000102,0.011199,0.011301,1.407819,0.015910,0.011301,0.000102,0.011199,0.011301
3,192.168.117.128,192.168.117.2,40736.0,53.0,Tunnel,0,3.532573,26,0,1.0000,...,0.001199,0.000000,0.000848,0.000848,1.414214,0.001199,0.000848,0.000000,0.000848,0.000848
4,192.168.117.128,192.168.117.2,43239.0,53.0,Tunnel,0,3.532573,26,0,1.0000,...,0.001403,0.000000,0.000992,0.000992,1.414214,0.001403,0.000992,0.000000,0.000992,0.000992


In [123]:
#check type of features 
print(f"Feature types:\n{X_train.dtypes}")

Feature types:
Character_frequency_entropy           float64
Domain_name_length                    float64
Non_printable_character_count         float64
Digit_letter_ratio                    float64
Unique_subdomains_per_flow            float64
NXDomain_ratio                        float64
Query_rate_per_sec                    float64
Response_size_variance                float64
Duration                              float64
FlowBytesSent                         float64
PacketLengthMean                      float64
PacketLengthMode                      float64
PacketLengthSkewFromMedian            float64
PacketLengthSkewFromMode              float64
PacketLengthCoefficientofVariation    float64
PacketTimeMedian                      float64
PacketTimeSkewFromMedian              float64
PacketTimeCoefficientofVariation      float64
dtype: object


In [124]:
# Remove duplicates
initial_rows = df.shape[0]
df = df.drop_duplicates()
duplicate_rows = initial_rows - df.shape[0]
print(f'\nRemoved duplicates: {duplicate_rows} rows')


Removed duplicates: 0 rows


In [125]:
initial_rows = df.shape[0]
missing_counts = df.isnull().sum()
print(missing_counts)

SourceIP                              10
DestIP                                10
SourcePort                             1
DestPort                               1
Label                                  0
WindowID                               0
Character_frequency_entropy            0
Domain_name_length                     0
Non_printable_character_count          0
Digit_letter_ratio                     0
Unique_subdomains_per_flow             0
NXDomain_ratio                         0
Query_rate_per_sec                     0
Response_size_variance                 0
Duration                               0
FlowBytesSent                          0
FlowSentRate                           0
FlowBytesReceived                      0
FlowReceivedRate                       0
PacketLengthVariance                   0
PacketLengthStandardDeviation          0
PacketLengthMean                       0
PacketLengthMedian                     0
PacketLengthMode                       0
PacketLengthSkew

In [126]:
cols_to_check = ['PacketLengthSkewFromMedian', 'PacketLengthSkewFromMode']
for col in cols_to_check:
    nulls = df[col].isnull().sum()
    infs = np.isinf(df[col]).sum()
    print(f"{col} -> Nulls: {nulls}, Infs: {infs}")

PacketLengthSkewFromMedian -> Nulls: 0, Infs: 0
PacketLengthSkewFromMode -> Nulls: 0, Infs: 0


In [127]:
def get_feature_summary(df, columns=None):
    """
    Returns a summary (min, max, mean) for specified columns in a DataFrame.
    If no columns are provided, it uses all numeric columns in the DataFrame.
    """
    if columns is None:
        columns = df.columns
    
    # Filter for columns that actually exist in the DataFrame
    existing_cols = [col for col in columns if col in df.columns]
    
    # Select only numeric columns for these specific statistics
    numeric_df = df[existing_cols].select_dtypes(include=['number'])
    
    if numeric_df.empty:
        return "No numeric columns found to summarize."
    
    # Calculate min, max, and mean
    summary = numeric_df.agg(['min', 'max', 'mean']).T
    
    # Optional: Rename columns for better readability
    summary.columns = ['Minimum', 'Maximum', 'Mean']
    
    return summary

# Example usage with your column list:
cols_to_summary = [
    'Character_frequency_entropy', 'Domain_name_length',
    'Non_printable_character_count', 'Digit_letter_ratio', 'Unique_subdomains_per_flow', 
    'NXDomain_ratio', 'Query_rate_per_sec', 'Response_size_variance', 'Duration', 
    'FlowBytesSent', 'FlowSentRate', 'FlowBytesReceived', 'FlowReceivedRate', 
    'PacketLengthVariance', 'PacketLengthStandardDeviation', 'PacketLengthMean',
    'PacketLengthMedian', 'PacketLengthMode', 'PacketLengthSkewFromMedian',
    'PacketLengthSkewFromMode', 'PacketLengthCoefficientofVariation',
    'PacketTimeStandardDeviation', 'PacketTimeMedian', 'PacketTimeSkewFromMedian', 
    'PacketTimeSkewFromMode', 'PacketTimeCoefficientofVariation', 
    'ResponseTimeTimeStandardDeviation', 'ResponseTimeTimeMean', 'ResponseTimeTimeMedian',
    'ResponseTimeTimeSkewFromMedian', 'ResponseTimeTimeSkewFromMode'
]

df_summary = get_feature_summary(df, cols_to_summary)
print(df_summary)


                                       Minimum       Maximum         Mean
Character_frequency_entropy           1.664498  5.175529e+00     3.525327
Domain_name_length                    5.000000  2.530000e+02    59.592996
Non_printable_character_count         0.000000  1.290000e+03     6.911882
Digit_letter_ratio                    0.000000  6.181818e+00     0.365938
Unique_subdomains_per_flow            1.000000  1.000000e+01     4.592369
NXDomain_ratio                        0.000000  0.000000e+00     0.000000
Query_rate_per_sec                    0.000019  1.000000e+04    19.384107
Response_size_variance                0.000000  3.547704e+05  9190.219376
Duration                              0.001000  5.403724e+05  4150.318840
FlowBytesSent                        65.000000  4.337000e+03   631.072276
FlowSentRate                          0.001609  1.064000e+06  3241.409311
FlowBytesReceived                    65.000000  4.337000e+03   631.072276
FlowReceivedRate                      

## STEP 3: OUTLIER & SKEW TREATMENT (after train/test split, NO DATA LEAKAGE)

In [128]:
# Split features and target
X = df.drop('Label', axis=1)   
y = df['Label']
print(f'\nFeatures shape: {X.shape}, Target shape: {y.shape}')

print('\n' + '=' * 60)
print('STEP 2: TRAIN / TEST SPLIT (70/30)')
print('=' * 60)

# First split: 70% train, 30% test
X_train, X_test, y_train, y_test= train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f'\nTrain: {X_train.shape[0]} rows ({100*len(X_train)/len(X):.0f}%)')
print(f'Test:  {X_test.shape[0]} rows ({100*len(X_test)/len(X):.0f}%)')

print(f'\nTrain class balance: {y_train.value_counts()[0]} benign, {y_train.value_counts()[1]} malicious')
print(f'Test class balance:  {y_test.value_counts()[0]} benign, {y_test.value_counts()[1]} malicious')


Features shape: (287196, 36), Target shape: (287196,)

STEP 2: TRAIN / TEST SPLIT (70/30)

Train: 201037 rows (70%)
Test:  86159 rows (30%)

Train class balance: 145148 benign, 55889 malicious
Test class balance:  62207 benign, 23952 malicious


In [129]:
to_exclude={'SourcePort','DestPort','WindowID'}
selected_columns = [col for col in numeric_cols if col not in to_exclude]
log1p_features=[]
for col in selected_columns:
    # Optional safety check: Ensure we only log non-negative values
    if X_train[col].min() >= 0:
        X_train[col] = np.log1p(X_train[col])
        X_test[col] = np.log1p(X_test[col])
        log1p_features.append(col)
    else:
        print(f"Skipping log1p for {col} because it contains negative values.")


Skipping log1p for PacketLengthSkewFromMedian because it contains negative values.
Skipping log1p for PacketLengthSkewFromMode because it contains negative values.
Skipping log1p for PacketTimeSkewFromMedian because it contains negative values.
Skipping log1p for PacketTimeSkewFromMode because it contains negative values.
Skipping log1p for ResponseTimeTimeSkewFromMedian because it contains negative values.
Skipping log1p for ResponseTimeTimeSkewFromMode because it contains negative values.


In [130]:
df_summary2 = get_feature_summary(X_train, selected_columns)
print(df_summary2)


                                       Minimum       Maximum        Mean
Character_frequency_entropy           1.044639      1.820594    1.503376
Domain_name_length                    1.791759      5.537334    3.535010
Non_printable_character_count         0.000000      7.163172    0.182966
Digit_letter_ratio                    0.000000      1.971553    0.216347
Unique_subdomains_per_flow            0.693147      2.397895    1.672099
NXDomain_ratio                        0.000000      0.000000    0.000000
Query_rate_per_sec                    0.000019      9.210440    0.818229
Response_size_variance                0.000000     12.758460    6.095901
Duration                              0.001000     13.200016    5.512788
FlowBytesSent                         4.189655      8.363576    6.250186
FlowSentRate                          0.001608     13.853770    2.421641
FlowBytesReceived                     4.189655      8.363576    6.250186
FlowReceivedRate                      0.001608     

# STEP 4: FEATURE SELECTION & REDUCTION (computed on train only, NO DATA LEAKAGE)

In [131]:

numeric_features=selected_columns
X_train_numeric = X_train[numeric_features]
X_test_numeric = X_test[numeric_features]

# # Drop high-correlation duplicates (based on train only)
# corr_matrix = X_train_numeric.corr().abs()
# upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
# X_train_numeric = X_train_numeric.drop(columns=to_drop)
# X_test_numeric = X_test_numeric.drop(columns=to_drop)

# print(f'Dropped {len(to_drop)} high-corr features')
# print(to_drop)
# print(f'Features after dropping: {X_train_numeric.shape[1]}')

In [132]:
X_train_numeric.columns

Index(['Character_frequency_entropy', 'Domain_name_length',
       'Non_printable_character_count', 'Digit_letter_ratio',
       'Unique_subdomains_per_flow', 'NXDomain_ratio', 'Query_rate_per_sec',
       'Response_size_variance', 'Duration', 'FlowBytesSent', 'FlowSentRate',
       'FlowBytesReceived', 'FlowReceivedRate', 'PacketLengthVariance',
       'PacketLengthStandardDeviation', 'PacketLengthMean',
       'PacketLengthMedian', 'PacketLengthMode', 'PacketLengthSkewFromMedian',
       'PacketLengthSkewFromMode', 'PacketLengthCoefficientofVariation',
       'PacketTimeStandardDeviation', 'PacketTimeMedian',
       'PacketTimeSkewFromMedian', 'PacketTimeSkewFromMode',
       'PacketTimeCoefficientofVariation', 'ResponseTimeTimeStandardDeviation',
       'ResponseTimeTimeMean', 'ResponseTimeTimeMedian',
       'ResponseTimeTimeSkewFromMedian', 'ResponseTimeTimeSkewFromMode'],
      dtype='object')

In [133]:
# Identify columns with at least one null value
null_columns = X_train_numeric.columns[X_train_numeric.isnull().any()]

if not null_columns.empty:
    print("Columns with null values and their counts:")
    print(X_train_numeric[null_columns].isnull().sum())
else:
    print("No null values found in X_train_numeric.")


No null values found in X_train_numeric.


In [134]:
X_train=X_train_numeric
X_test=X_test_numeric

In [135]:
print('\n' + '=' * 60)
print('STEP 5: FEATURE SCALING (RobustScaler)')
print('=' * 60)
y_train = y_train.map({'Normal': 0, 'Tunnel': 1})
y_test=y_test.map({'Normal': 0, 'Tunnel': 1})

# RobustScaler uses IQR (Interquartile Range) - resistant to outliers
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_train.columns)

print(f'Scaled train - median: {X_train_scaled.values.mean():.4f}, IQR preserved')
print('RobustScaler fit on train set only (no data leakage)')

#save the scaler for later use
scaler_path = Path('../models/robust_scaler.pkl')
scaler_path.parent.mkdir(parents=True, exist_ok=True)
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved RobustScaler to {scaler_path}')


STEP 5: FEATURE SCALING (RobustScaler)


Scaled train - median: 0.3808, IQR preserved
RobustScaler fit on train set only (no data leakage)
Saved RobustScaler to ..\models\robust_scaler.pkl


In [136]:
print('\n' + '=' * 60)
print('STEP 6: CLASS IMBALANCE HANDLING (SMOTE)')
print('=' * 60)

print('\nBefore SMOTE:')
print(f'  Benign: {(y_train == 0).sum():,}')
print(f'  Malicious: {(y_train == 1).sum():,}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print('\nAfter SMOTE:')
print(f'  Benign: {(y_train_balanced == 0).sum():,}')
print(f'  Malicious: {(y_train_balanced == 1).sum():,}')

X_train_balanced = pd.DataFrame(X_train_balanced, columns=X_train.columns)


STEP 6: CLASS IMBALANCE HANDLING (SMOTE)

Before SMOTE:
  Benign: 145,148
  Malicious: 55,889

After SMOTE:
  Benign: 145,148
  Malicious: 145,148


In [137]:
print('\n' + '=' * 60)
print('PREPROCESSING VERIFICATION')
print('=' * 60)

print(f'\nTrain (SMOTE balanced): {X_train_balanced.shape}')
print(f'Test (original): {X_test_scaled.shape}')
print(f'\nNull values in train: {X_train_balanced.isnull().sum().sum()}')
print(f'Null values in test: {X_test_scaled.isnull().sum().sum()}')



PREPROCESSING VERIFICATION

Train (SMOTE balanced): (290296, 31)
Test (original): (86159, 31)

Null values in train: 0
Null values in test: 0


In [138]:
X_train_balanced.columns

Index(['Character_frequency_entropy', 'Domain_name_length',
       'Non_printable_character_count', 'Digit_letter_ratio',
       'Unique_subdomains_per_flow', 'NXDomain_ratio', 'Query_rate_per_sec',
       'Response_size_variance', 'Duration', 'FlowBytesSent', 'FlowSentRate',
       'FlowBytesReceived', 'FlowReceivedRate', 'PacketLengthVariance',
       'PacketLengthStandardDeviation', 'PacketLengthMean',
       'PacketLengthMedian', 'PacketLengthMode', 'PacketLengthSkewFromMedian',
       'PacketLengthSkewFromMode', 'PacketLengthCoefficientofVariation',
       'PacketTimeStandardDeviation', 'PacketTimeMedian',
       'PacketTimeSkewFromMedian', 'PacketTimeSkewFromMode',
       'PacketTimeCoefficientofVariation', 'ResponseTimeTimeStandardDeviation',
       'ResponseTimeTimeMean', 'ResponseTimeTimeMedian',
       'ResponseTimeTimeSkewFromMedian', 'ResponseTimeTimeSkewFromMode'],
      dtype='object')

In [139]:
print('\n' + '=' * 60)
print('STEP 7: SAVING PREPROCESSED DATA')
print('=' * 60)

train_df = X_train_balanced.copy()

test_df = X_test_scaled.copy()

train_output = Path('../data/train_data.csv')
test_output = Path('../data/test_data.csv')

train_df.to_csv(train_output, index=False)
test_df.to_csv(test_output, index=False)

print(f'✓ Train data: {train_df.shape[0]:,} rows → {train_output.name}')
print(f'✓ Test data:  {test_df.shape[0]:,} rows → {test_output.name}')


STEP 7: SAVING PREPROCESSED DATA
✓ Train data: 290,296 rows → train_data.csv
✓ Test data:  86,159 rows → test_data.csv


In [140]:
print('\n' + '=' * 60)
print('STEP 8: SAVING PREPROCESSING METADATA')
print('=' * 60)
#TO DO : save the log transform on the selected features
with open('../data/log1p_features.txt', 'w') as f:
    f.write('\n'.join(log1p_features))
print(f'✓ Log-transformed feature list saved ({len(log1p_features)} features)')
# Save feature names
with open('../data/feature_names.txt', 'w') as f:
    f.write('\n'.join(X_train.columns.tolist()))
print(f'✓ Feature names saved ({len(X_train.columns)} features)')




STEP 8: SAVING PREPROCESSING METADATA
✓ Log-transformed feature list saved (25 features)
✓ Feature names saved (31 features)
